In [1]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

import numpy as np
import pandas as pd

data = pd.read_csv("data/classification_df.csv")
data['final_result'] = data['final_result'].apply(lambda x: 1 if x == 'Withdrawn' else 0)
data = data.rename(columns={'final_result':'Target'})

data.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,Target,date_registration,date_unregistration,score,is_banked,total_clicks,activity_diversity
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,0,-159.0,NaN,82.4,0,934.0,6.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,0,-53.0,NaN,65.4,0,1435.0,7.0
2,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,0,-52.0,NaN,76.3,0,2158.0,8.0
3,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,0,-176.0,NaN,55.0,0,1034.0,7.0
4,AAA,2013J,38053,M,Wales,A Level or Equivalent,80-90%,35-55,0,60,N,0,-110.0,NaN,66.9,0,2445.0,8.0


In [2]:
X = data.drop(["Target","code_module","code_presentation","id_student","date_unregistration"], axis=1)
y = data["Target"]
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
x_train

,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,date_registration,score,is_banked,total_clicks,activity_diversity
15121,F,South East Region,Lower Than A Level,80-90%,0-35,1,60,Y,-94.0,20.150,1,73.0,6.0
12216,F,South West Region,Lower Than A Level,30-40%,0-35,0,60,N,-116.0,121.025,0,1524.0,9.0
12054,M,South East Region,No Formal quals,20-30%,0-35,0,60,N,-122.0,142.750,0,1005.0,10.0
18385,M,South Region,A Level or Equivalent,70-80%,0-35,0,60,N,-55.0,90.000,0,6163.0,14.0
22117,M,North Region,Lower Than A Level,NaN,35-55,1,60,N,-43.0,80.000,0,3022.0,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21575,M,Wales,A Level or Equivalent,80-90%,0-35,0,60,N,-34.0,94.750,0,9816.0,14.0
5390,F,Wales,Lower Than A Level,0-10%,35-55,0,60,N,-113.0,66.050,0,1201.0,8.0
860,M,Scotland,HE Qualification,20-30%,0-35,0,120,N,-64.0,78.910,0,465.0,7.0
15795,M,West Midlands Region,A Level or Equivalent,30-40%,0-35,0,60,N,-56.0,66.360,0,1049.0,10.0


In [3]:
class Preparator(BaseEstimator,TransformerMixin):
    def __init__(self):
        self.medians = {}
        self.modes = {}
        pass
    def fit(self, X:pd.DataFrame, y= None):

        for column in X:
            try:
                self.medians[column] = X[column].median()
            except:
                pass
            finally:
                self.modes[column] = X[column].mode()[0]
        return self
    def transform(self, X:pd.DataFrame):
        for column in X:
            try:
                X[column] =  X[column].fillna(self.medians[column])
            except:
                pass
            finally:
                X[column] = X[column].fillna(self.modes[column])
        X['gender'] = X['gender'].apply(lambda x: 1 if x == 'M' else 0)
        X['disability'] = X['disability'].apply(lambda x: 1 if x == 'Y' else 0)
        result = pd.get_dummies(X,columns=["region","age_band","imd_band"],dtype=np.int8)
        edu_order = {"No Formal quals":0,"Lower Than A Level":1,"A Level or Equivalent":2,"HE Qualification":3,"Post Graduate Qualification":4}

        result["highest_education"] = result["highest_education"].map(edu_order)
        return result





In [4]:
preproc = Preparator()
dfset = preproc.fit_transform(X=x_train)
dfset.head()

,gender,highest_education,num_of_prev_attempts,studied_credits,disability,date_registration,score,is_banked,total_clicks,activity_diversity,...,imd_band_0-10%,imd_band_10-20,imd_band_20-30%,imd_band_30-40%,imd_band_40-50%,imd_band_50-60%,imd_band_60-70%,imd_band_70-80%,imd_band_80-90%,imd_band_90-100%
15121,0,1,1,60,1,-94.0,20.150,1,73.0,6.0,...,0,0,0,0,0,0,0,0,1,0
12216,0,1,0,60,0,-116.0,121.025,0,1524.0,9.0,...,0,0,0,1,0,0,0,0,0,0
12054,1,0,0,60,0,-122.0,142.750,0,1005.0,10.0,...,0,0,1,0,0,0,0,0,0,0
18385,1,2,0,60,0,-55.0,90.000,0,6163.0,14.0,...,0,0,0,0,0,0,0,1,0,0
22117,1,1,1,60,0,-43.0,80.000,0,3022.0,12.0,...,0,0,0,1,0,0,0,0,0,0


In [5]:
pipeline = Pipeline([
    ("prep", Preparator()),
    ("clf", LogisticRegression(solver="liblinear"))
]
)
pipeline.fit(X=x_train, y=y_train)
y_ped = pipeline.predict(x_test)

print(classification_report(y_test,y_ped))

              precision    recall  f1-score   support

           0       0.88      0.94      0.91      4204
           1       0.62      0.42      0.50       965

    accuracy                           0.84      5169
   macro avg       0.75      0.68      0.70      5169
weighted avg       0.83      0.84      0.83      5169



In [7]:
pipeline2 = Pipeline([
    ("prep", Preparator()),
    ("clf", GradientBoostingClassifier()),
]
)
pipeline2.fit(X=x_train, y=y_train)
y_ped = pipeline2.predict(x_test)

print(classification_report(y_test,y_ped))

              precision    recall  f1-score   support

           0       0.92      0.91      0.92      4204
           1       0.63      0.68      0.65       965

    accuracy                           0.86      5169
   macro avg       0.78      0.79      0.78      5169
weighted avg       0.87      0.86      0.87      5169



In [9]:
voting_classifier = VotingClassifier([
    ("lgb", LogisticRegression(solver="liblinear")),
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("ada", AdaBoostClassifier()),
    ("gb", GradientBoostingClassifier()),

], voting='soft')

pipeline3 = Pipeline([
    ("prep", Preparator()),
    ("clf", voting_classifier),
]
)

pipeline3.fit(X=x_train, y=y_train)
y_ped2 = pipeline3.predict(x_test)
print(classification_report(y_test,y_ped2))

              precision    recall  f1-score   support

           0       0.90      0.93      0.91      4204
           1       0.63      0.54      0.58       965

    accuracy                           0.86      5169
   macro avg       0.76      0.73      0.75      5169
weighted avg       0.85      0.86      0.85      5169

